## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [52]:
import os
from dotenv import load_dotenv
load_dotenv() ## loading all the environment variables

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10a076e40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10a077b60>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage
result = model.invoke([HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")])
result

AIMessage(content='Hello Krish! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether you’re looking to brainstorm ideas, dive into technical details, or just chat about AI trends, I’m here to help.', additional_kwargs={'reasoning_content': 'The user says: "Hi , My name is Krish and I am a Chief AI Engineer". The user likely wants a greeting or maybe a conversation. We should respond politely, acknowledging. Possibly ask how we can assist. The user might want to talk about AI. We should respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 85, 'total_tokens': 203, 'completion_time': 0.127895794, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.005543962, 'prompt_tokens_details': None, 'queue_time': 0.346732906, 'total_time': 0.133439756}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_

In [5]:
result.content

'Hello Krish! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether you’re looking to brainstorm ideas, dive into technical details, or just chat about AI trends, I’m here to help.'

In [7]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer"),
        AIMessage(content="Hello Krish! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You’re Krish, and you’re a Chief AI Engineer—leading AI strategy, architecture, and development for your organization.', additional_kwargs={'reasoning_content': 'User: "Hey What\'s my name and what do I do?" We have earlier conversation: "Hi , My name is Krish and I am a Chief AI Engineer". So answer: name Krish, role Chief AI Engineer. Need to respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 152, 'total_tokens': 238, 'completion_time': 0.088199268, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.007337702, 'prompt_tokens_details': None, 'queue_time': 0.334675652, 'total_time': 0.09553697}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1074f9ce08', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10b3a-2454-7d50-a7ce-d4370061a6b0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [11]:
config={"configurable":{"session_id":"chat1"}}    #This is configuration information for RunnableWithMessageHistory.

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")],
    config=config
)

In [14]:
response.content

'Nice to meet you, Krish! 🚀  \nAs a Chief AI Engineer, you’re probably juggling a lot—models, pipelines, team dynamics, or maybe the latest research. How can I support you today? Whether it’s a quick question, a deep dive, or just brainstorming, I’m here to help.'

In [15]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='You introduced yourself as Krish.', additional_kwargs={'reasoning_content': 'User says: "What\'s my name?" They previously introduced themselves as Krish. So answer: Krish.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 252, 'total_tokens': 291, 'completion_time': 0.039850614, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.01217447, 'prompt_tokens_details': None, 'queue_time': 0.315135132, 'total_time': 0.052025084}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10b6a-5497-7203-a07b-04af6cd0348f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 252, 'output_tokens': 39, 'total_tokens': 291, 'output_token_details': {'reasoning': 23}})

In [16]:
## change the config-->session id
config2={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config2
)
response.content

'I’m not sure what your name is. Could you let me know?'

In [18]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config2
)
response.content

'Nice to meet you, John! How can I help you today?'

In [19]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config2
)
response.content

'You just told me your name is John. If I’m mistaken, just let me know!'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [20]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [21]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")]})

AIMessage(content='Hello Krish! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says "Hi My name is Krish". They likely want a greeting and maybe ask for something. We need to respond politely, greet them, ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 98, 'total_tokens': 157, 'completion_time': 0.06347441, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.004716617, 'prompt_tokens_details': None, 'queue_time': 0.305910089, 'total_time': 0.068191027}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10c14-9f9f-7c52-bc0e-6aea9a615ef8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 59, 'total_tokens': 157, 'output_token_details': {'reasoning': 37}})

In [23]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [25]:
config3 = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config=config3
)

response

AIMessage(content='Nice to meet you, Krish! How can I help you today?', additional_kwargs={'reasoning_content': 'User repeats greeting. Need to respond.'}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 127, 'total_tokens': 160, 'completion_time': 0.034276319, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.007174399, 'prompt_tokens_details': None, 'queue_time': 0.046445488, 'total_time': 0.041450718}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10c56-a31b-7020-baff-2690bfa6b6e9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 33, 'total_tokens': 160, 'output_token_details': {'reasoning': 9}})

In [26]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config3,
)

response.content

'Your name is Krish.'

In [27]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [28]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"Hindi"})
response.content

'नमस्ते कृष्ण जी! आपका स्वागत है। मैं आपकी कैसे सहायता कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [31]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [32]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते, श्री कृष्ण! आपको कैसे सहायता कर सकता हूँ?'

In [33]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [34]:
response.content

'आपका नाम कृष्ण है। यदि आप कुछ और पूछना चाहें तो बताइए!'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [46]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [47]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I’m not sure—what’s your favorite flavor or type of ice cream? (e.g., classic vanilla, chocolate, mint chocolate chip, or maybe something more adventurous like salted caramel or a dairy‑free option?) Let me know and I can suggest something you’ll love!'

In [48]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked: “whats 2 + 2.”'

In [49]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

/Users/vinaykumarranjan/Documents/Langchain/cenv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [50]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'I don’t actually know your name—could you tell me?'

In [51]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'I’m not sure—there wasn’t a math problem mentioned in our conversation so far. If you have one in mind, feel free to share it and I’ll help you out!'

                     USER INPUT
                         │
                         ▼
              RunnableWithMessageHistory
                         │
                    session_id
                     "chat5"
                         │
                         ▼
                 ChatMessageHistory
                         │
                    conversation
                         │
                         ▼
                ┌─────────────────┐
                │   trim_messages │
                │                 │
                │ max_tokens=45   │
                │ strategy=last   │
                │ include_system  │
                └────────┬────────┘
                         │
                  trimmed messages
                         │
                         ▼
               MessagesPlaceholder
                         │
                         ▼
                ChatPromptTemplate
                         │
                         ▼
                       Model
                         │
                         ▼
                     AIMessage

There are three separate jobs here:

1. RunnableWithMessageHistory

"Which conversation should I remember?"
session_id = "chat5"
store["chat5"]

2. trim_messages

"How much of that conversation should I send to the model?"
max_tokens=45
strategy="last"
Keep recent messages

3. MessagesPlaceholder

"Where should those messages go in my prompt?"
MessagesPlaceholder(variable_name="messages")
Insert messages here

So the mental model is:
SESSION
  ↓
"chat5"
  ↓
retrieve conversation
  ↓
TRIM conversation
  ↓
put messages into placeholder
  ↓
PROMPT
  ↓
LLM
  ↓
RESPONSE